# Chapter 3 — Hedging Strategies Using Futures

Code behind every worked example in the chapter. **Standard library only.**

| Section | What it reproduces |
|---|---|
| 1 | An oil producer's short hedge (Example 3.1) |
| 2 | Basis risk in numbers (Example 3.2) |
| 3 | Hedging jet fuel with heating oil (Example 3.3) |
| 4 | The hedge ratio depends on the horizon (Example 3.4) |
| 5 | Hedging a portfolio (Example 3.5) |
| 6 | A rolling hedge, roll by roll (Example 3.6) |
| 7 | The cost of rolling (Example 3.7) |

Technical Note 3 develops the minimum-variance ratio further — the multivariate
hedge, what estimation costs, and the tailing adjustment. Its arithmetic is in
`tools/verify/ch03-hedge-ratio.py`.

Each cell prints the computed value beside the one the chapter prints.

In [1]:
def show(label, got, printed, tol=6e-4):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %14.4f   book %12.4f   %s" % (label, got, printed, flag))

## 1. A short hedge — Example 3.1

The producer sells the physical at whatever spot turns out to be, and the futures
leg makes up the difference. The effective price lands near the futures price at
which the hedge was placed — near, but not on it, and the ten cents is the whole
subject of the next section.

In [2]:
BARRELS = 1_000_000
spot_0, fut_0 = 80.00, 80.50
spot_1, fut_1 = 72.00, 72.10

revenue = spot_1 * BARRELS
gain = (fut_0 - fut_1) * BARRELS
show("revenue from selling the oil", revenue, 72_000_000)
show("gain on the futures position", gain, 8_400_000)
show("total", revenue + gain, 80_400_000)
show("effective price per barrel", (revenue + gain) / BARRELS, 80.40)
show("shortfall against the futures price", fut_0 - (revenue + gain) / BARRELS, 0.10)

revenue from selling the oil                    72000000.0000   book 72000000.0000   ok
gain on the futures position                     8400000.0000   book 8400000.0000   ok
total                                           80400000.0000   book 80400000.0000   ok
effective price per barrel                            80.4000   book      80.4000   ok
shortfall against the futures price                    0.1000   book       0.1000   ok


## 2. Basis risk — Example 3.2

The effective price is $F_1 + b_2$: the futures price locked in, plus whatever the
basis turns out to be when the hedge is lifted. Only $b_2$ is unknown at the
outset, so the basis is exactly the risk the hedge does not remove.

In [3]:
S1, F1 = 62.00, 60.00
S2, F2 = 58.00, 57.50
b1, b2 = S1 - F1, S2 - F2
show("basis at inception b1", b1, 2.00)
show("basis when lifted b2", b2, 0.50)
show("effective price F1 + b2", F1 + b2, 60.50)
show("had the basis held at 2.00", F1 + b1, 62.00)
show("basis weakening, the loss per unit", b1 - b2, 1.50)

show("spot fell by", S1 - S2, 4.00)
show("fraction of the move left unhedged, %", 100 * (b1 - b2) / (S1 - S2), 37.5)

basis at inception b1                                  2.0000   book       2.0000   ok
basis when lifted b2                                   0.5000   book       0.5000   ok
effective price F1 + b2                               60.5000   book      60.5000   ok
had the basis held at 2.00                            62.0000   book      62.0000   ok
basis weakening, the loss per unit                     1.5000   book       1.5000   ok
spot fell by                                           4.0000   book       4.0000   ok
fraction of the move left unhedged, %                 37.5000   book      37.5000   ok


## 3. Cross hedging — Example 3.3

No jet fuel contract trades, so the airline uses heating oil. The minimum-variance
ratio is well below one, and the residual is large: with $\rho = 0.8$ roughly a
third of the variance survives the hedge.

In [4]:
sigma_S, sigma_F, rho = 0.032, 0.040, 0.80
GALLONS, PER_CONTRACT = 2_000_000, 42_000

h = rho * sigma_S / sigma_F
N = h * GALLONS / PER_CONTRACT
show("h* = rho sigma_S / sigma_F", h, 0.64)
show("N* = h* Q_A / Q_F", N, 30.5, tol=2e-3)
N_bought = round(N)
show("contracts bought", N_bought, 30)
show("gallons covered, 30 x 42,000", N_bought * PER_CONTRACT, 1_260_000)
show("R^2 = rho^2, variance removed", rho ** 2, 0.64)
show("variance remaining, %", 100 * (1 - rho ** 2), 36.0)

h* = rho sigma_S / sigma_F                             0.6400   book       0.6400   ok
N* = h* Q_A / Q_F                                     30.4762   book      30.5000   ok
contracts bought                                      30.0000   book      30.0000   ok
gallons covered, 30 x 42,000                     1260000.0000   book 1260000.0000   ok
R^2 = rho^2, variance removed                          0.6400   book       0.6400   ok
variance remaining, %                                 36.0000   book      36.0000   ok


## 4. The horizon matters — Example 3.4

The same two series, estimated daily and monthly, give ratios nearly 40% apart. Neither
is wrong; each is right for a hedge held that long. Correlation between related
assets usually rises with the measurement horizon, so the ratio does too. The
rise comes mostly from the correlation (0.62 daily, 0.80 monthly) and partly from
the volatility ratio, which is 0.745 daily against 0.800 monthly.

In [5]:
rows = [("Daily changes",   0.0070, 0.0094, 0.62, 0.46),
        ("Monthly changes", 0.0320, 0.0400, 0.80, 0.64)]
hr = {}
for name, sS, sF, rr, printed in rows:
    hr[name] = rr * sS / sF
    show("%-16s h*" % name, hr[name], printed, tol=8e-3)

# The rise comes mostly from the correlation, partly from the volatility ratio.
show("daily volatility ratio", rows[0][1] / rows[0][2], 0.745)
show("monthly volatility ratio", rows[1][1] / rows[1][2], 0.800)
show("monthly over daily, %", 100 * (hr["Monthly changes"] / hr["Daily changes"] - 1), 38.6, tol=0.02)

Daily changes    h*                                    0.4617   book       0.4600   ok
Monthly changes  h*                                    0.6400   book       0.6400   ok
daily volatility ratio                                 0.7447   book       0.7450   ok
monthly volatility ratio                               0.8000   book       0.8000   ok
monthly over daily, %                                 38.6175   book      38.6000   ok


## 5. Hedging a portfolio — Example 3.5

Beta scales the exposure: a portfolio with $\beta = 1.5$ needs half again as many
contracts as its value alone would suggest. The offset is close but not exact,
because 36 contracts is not 36.07.

In [6]:
VALUE, BETA, INDEX, MULT = 5_050_000.0, 1.5, 4200.0, 50.0
contract = INDEX * MULT
show("value of one contract", contract, 210_000)
N_star = BETA * VALUE / contract
show("N* = beta * V / contract", N_star, 36.07, tol=3e-4)

N_traded = 36
index_1 = INDEX * 0.90
show("index after a 10% fall", index_1, 3780)
show("gain on the futures", N_traded * (INDEX - index_1) * MULT, 756_000)
show("portfolio fall, beta * 10%", VALUE * BETA * 0.10, 757_500)
show("residual", VALUE * BETA * 0.10 - N_traded * (INDEX - index_1) * MULT, 1_500)

value of one contract                             210000.0000   book  210000.0000   ok
N* = beta * V / contract                              36.0714   book      36.0700   ok
index after a 10% fall                              3780.0000   book    3780.0000   ok
gain on the futures                               756000.0000   book  756000.0000   ok
portfolio fall, beta * 10%                        757500.0000   book  757500.0000   ok
residual                                            1500.0000   book    1500.0000   ok


## 6. Rolling the hedge — Example 3.6

Each roll closes one contract and opens a dearer one, so a **short** hedger gains
in contango. The improvement decomposes exactly into the roll gains plus the
final basis, which is the check worth doing.

In [7]:
legs = [("A", 80.00, 78.00), ("B", 78.60, 76.50),
        ("C", 77.00, 75.00), ("D", 75.40, 74.00)]
total = 0.0
for name, opened, closed in legs:
    total += opened - closed
    show("gain on leg %s" % name, opened - closed, round(opened - closed, 2))
show("total futures gain", total, 7.50)

spot_final = 73.90
show("effective price", spot_final + total, 81.40)

roll_gains = sum(legs[i + 1][1] - legs[i][2] for i in range(len(legs) - 1))
final_basis = spot_final - legs[-1][2]
show("roll gains, summed", roll_gains, 1.50)
show("final basis", final_basis, -0.10)
show("80.00 + roll gains + final basis", legs[0][1] + roll_gains + final_basis, 81.40)

gain on leg A                                          2.0000   book       2.0000   ok
gain on leg B                                          2.1000   book       2.1000   ok
gain on leg C                                          2.0000   book       2.0000   ok
gain on leg D                                          1.4000   book       1.4000   ok
total futures gain                                     7.5000   book       7.5000   ok
effective price                                       81.4000   book      81.4000   ok
roll gains, summed                                     1.5000   book       1.5000   ok
final basis                                           -0.1000   book      -0.1000   ok
80.00 + roll gains + final basis                      81.4000   book      81.4000   ok


## 7. The cost of a stack — Example 3.7

Stacking a multi-year exposure in the front contract turns the roll into a
recurring bill. The direction depends on the curve shape, and the magnitude is
what makes it a treasury problem rather than a rounding error.

In [8]:
ROLLS, COST_PER_BBL, AVG_POSITION = 35, 0.15, 18_000_000
show("cumulative roll cost, $m", ROLLS * COST_PER_BBL * AVG_POSITION / 1e6, 94.5)

cumulative roll cost, $m                              94.5000   book      94.5000   ok
